# 🇿🇼 Sisonke E-Friend Model Fine-Tuning (QLoRA + Unsloth)

This notebook fine-tunes **Qwen2.5-1.5B-Instruct** into a dedicated **E-Friend** peer counseling model for youth in Zimbabwe using **Unsloth** on a free Google Colab T4 GPU (~20 mins).

### Pipeline Steps:
1. Install Unsloth and load quantized base model
2. Upload `sisonke_train.jsonl`
3. Fine-tune using QLoRA with ChatML formatting
4. Export directly to **GGUF format (`q4_k_m`)** for local Ollama and VPS deployment

In [ ]:
# Step 1: Install Unsloth & dependencies
!pip install --no-deps "xformers<0.0.29" "trl<0.9.0" peft accelerate bitsandbytes
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

In [ ]:
# Step 2: Load Base Model with 4-bit Quantization
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
load_in_4bit = True

# High-speed, multilingual base model
model_name = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    load_in_4bit = load_in_4bit,
)

# Attach LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("Model and LoRA adapters successfully initialized!")

In [ ]:
# Step 3: Upload and Prepare Training Data
from google.colab import files
import json
from datasets import Dataset

print("Please select and upload 'sisonke_train.jsonl':")
uploaded = files.upload()
filename = list(uploaded.keys())[0]

data = []
with open(filename, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            data.append(json.loads(line))

dataset = Dataset.from_list(data)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)
print(f"Prepared {len(dataset)} training examples.")

In [ ]:
# Step 4: Run Training
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 150,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("Starting training on free Colab T4 GPU...")
trainer_stats = trainer.train()
print("Training completed successfully!")

In [ ]:
# Step 5: Test the model locally in Colab
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": "You are E-Friend, a warm, caring peer supporter for youth in Zimbabwe on the Sisonke wellness app."},
    {"role": "user", "content": "I'm feeling very overwhelmed with exams and home issues today."}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(input_ids = inputs, max_new_tokens = 150, temperature = 0.4)
response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print("E-Friend Response:\n", response)

In [ ]:
# Step 6: Export to GGUF format for Ollama
print("Quantizing and converting model to GGUF (Q4_K_M)...")
model.save_pretrained_gguf("sisonke-efriend-1.5b", tokenizer, quantization_method = "q4_k_m")

# Download the GGUF file to your computer
import glob
gguf_files = glob.glob("sisonke-efriend-1.5b/*.gguf")
if gguf_files:
    print(f"Downloading {gguf_files[0]}...")
    files.download(gguf_files[0])
else:
    print("GGUF file saved in directory: sisonke-efriend-1.5b/")